<a href="https://colab.research.google.com/github/siri780/movielens-recommender/blob/main/movielens_recommender.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Movie recommendations on MovieLens 1M
Four recommenders, evaluated the way production teams do it: hide each user's most recent movie and check
whether the model ranks it in the top 10. Run everything with **Runtime → Run all** (about 2 minutes, CPU is fine).

## 1. Get the code and the data

In [1]:
!git clone -q https://github.com/siri780/movielens-recommender.git repo 2>/dev/null || (cd repo && git pull -q)
!curl -sL -o ml-1m.zip https://files.grouplens.org/datasets/movielens/ml-1m.zip && unzip -oq ml-1m.zip
import sys; sys.path[:0] = ["repo", "repo/src"]
import recsys
ratings, movies = recsys.load_movielens("ml-1m")
print(f"{len(ratings):,} ratings from {ratings.user.nunique():,} users on {ratings.item.nunique():,} movies")

1,000,209 ratings from 6,040 users on 3,706 movies


## 2. Leave-last-out split
Each user's most recent movie becomes the test item; everything before it is training data.

In [2]:
train, test = recsys.split_last_out(ratings)
X, test_u, test_i, items = recsys.to_matrix(train, test)
print(f"train matrix: {X.shape[0]:,} users x {X.shape[1]:,} movies, {X.nnz:,} interactions ({X.nnz/(X.shape[0]*X.shape[1]):.1%} filled)")

train matrix: 6,040 users x 3,704 movies, 994,169 interactions (4.4% filled)


## 3. Train and evaluate all four models

In [3]:
table, per_user = recsys.run(X, test_u, test_i)
table.round(4)

,HitRate@10,NDCG@10,coverage
model,,,
Popularity,0.0364,0.0179,0.0537
ItemKNN,0.0704,0.0365,0.1326
PureSVD,0.0972,0.0491,0.3296
EASE,0.1101,0.0545,0.3512


## 4. Is the best model really better? (bootstrap over users)

In [4]:
ranked = table["NDCG@10"].sort_values(ascending=False).index
for a, b in [(ranked[0], ranked[1]), (ranked[0], "Popularity")]:
    m, lo, hi = recsys.bootstrap_diff(per_user[a], per_user[b])
    print(f"{a} vs {b}: NDCG@10 {m:+.4f}  (95% CI {lo:+.4f} to {hi:+.4f})")

EASE vs PureSVD: NDCG@10 +0.0054  (95% CI +0.0029 to +0.0080)
EASE vs Popularity: NDCG@10 +0.0367  (95% CI +0.0321 to +0.0412)


## 5. Example: what EASE recommends for one user

In [5]:
import numpy as np
title = movies.set_index("item")["title"]
u = test_u[0]
watched = train[train.user == train.user.unique()[u]].sort_values("ts").tail(5)["item"]
scores = recsys.ease(X, np.array([u]))[0]
scores[X[u].toarray()[0] > 0] = -np.inf
top = items[np.argsort(-scores)[:10]]
print("Recently watched:"); print("  " + "\n  ".join(title[watched].tolist()))
print("\nRecommended:");   print("  " + "\n  ".join(title[top].tolist()))
print("\nHidden next movie:", title[items[test_i[0]]])

Recently watched:
  Bug's Life, A (1998)
  Antz (1998)
  Hunchback of Notre Dame, The (1996)
  Hercules (1997)
  Mulan (1998)

Recommended:
  Lion King, The (1994)
  Shawshank Redemption, The (1994)
  Babe (1995)
  Wrong Trousers, The (1993)
  Little Mermaid, The (1989)
  Sleeping Beauty (1959)
  Star Wars: Episode V - The Empire Strikes Back (1980)
  Fantasia (1940)
  American Beauty (1999)
  Pinocchio (1940)

Hidden next movie: Pocahontas (1995)
